Check existing tensorflow and keras library version using the following command
```
!pip show tensorflow
!pip show keras
```

Upgrade tensorflow and keras library to latest version using the following command
```
!pip install --upgrade tensorflow
!pip install --upgrade keras
```

## Importing dependencies

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Flatten
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

ModuleNotFoundError: No module named 'pandas'

## Loading dataset

In [2]:
data = pd.read_csv("/kaggle/input/diabetic-retinopathy-resized/trainLabels_cropped.csv")
print(data.value_counts().sum())
print(data['level'].unique())
print(data['level'].value_counts().unique())
data.head()

35108
[0 1 2 4 3]
[25802  5288  2438   872   708]


,Unnamed: 0.1,Unnamed: 0,image,level
0,0,0,10_left,0
1,1,1,10_right,0
2,2,2,13_left,0
3,3,3,13_right,0
4,4,4,15_left,1


## Dropping unnecessary columns

In [3]:
data.drop(columns=['Unnamed: 0.1','Unnamed: 0'], inplace=True)
data.head()

,image,level
0,10_left,0
1,10_right,0
2,13_left,0
3,13_right,0
4,15_left,1


## Creating a balanced dataset

In [4]:
from imblearn.under_sampling import RandomUnderSampler
import numpy as np

# Define your dataset
X = data['image']  # Your feature data
y = data['level']  # Your class labels

# Convert DataFrame or Series to a numpy array
X_array = X.values.reshape(-1,1)
y_array = y.values.reshape(-1,1)

# Count the number of datapoints for each class
class_counts = {0: 25802, 1: 5288, 2: 2438, 3: 708, 4: 872}

# Find the target number of datapoints
target_count = class_counts[3]

# Undersample the majority classes
undersampler = RandomUnderSampler(sampling_strategy={label: target_count for label in class_counts.keys() if label != 3})
X_resampled, y_resampled = undersampler.fit_resample(X_array, y_array)

# Check the new class counts
unique, counts = np.unique(y_resampled, return_counts=True)
resampled_class_counts = dict(zip(unique, counts))
print("Resampled class counts:", resampled_class_counts)

Resampled class counts: {0: 708, 1: 708, 2: 708, 3: 872, 4: 708}


In [5]:
new_X = []
for i in X_resampled:
  for j in i:
    new_X.append(j)

In [6]:
y_resampled

array([0, 0, 0, ..., 4, 4, 4])

In [7]:
# Create a new dataframe with resampled data
resampled_data = {'image': new_X,
        'level': y_resampled}

balanced_data = pd.DataFrame(resampled_data)

# Display the first few rows of the new dataframe
balanced_data.sample(10)

,image,level
3483,4909_right,4
1515,26630_left,2
3113,11032_right,4
2174,2901_right,3
2140,932_left,3
2407,15450_left,3
618,15431_left,0
20,34800_right,0
490,19518_left,0
2074,42973_left,2


In [8]:
print(balanced_data.value_counts().sum())
print(balanced_data['level'].unique())
print(balanced_data['level'].value_counts().unique())

3704
[0 1 2 3 4]
[872 708]


In [9]:
balanced_data['image'] = balanced_data['image']+'.jpeg'
balanced_data.head()

,image,level
0,28956_left.jpeg,0
1,44150_left.jpeg,0
2,2239_right.jpeg,0
3,23593_left.jpeg,0
4,18556_left.jpeg,0


### Splitting data

In [10]:
train, val = train_test_split(balanced_data, test_size=0.2, random_state=42)

In [11]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 5

### Rescaling image data

In [12]:
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

In [13]:
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train,
    directory='../input/diabetic-retinopathy-resized/resized_train_cropped/resized_train_cropped/',
    x_col='image',
    y_col='level',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='raw')

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val,
    directory='../input/diabetic-retinopathy-resized/resized_train_cropped/resized_train_cropped/',
    x_col='image',
    y_col='level',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='raw')

Found 2963 validated image filenames.
Found 741 validated image filenames.


### Loading MobileNetV2

In [14]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [15]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

In [16]:
model = Model(inputs=base_model.input, outputs=predictions)

In [17]:
for layer in base_model.layers:
    layer.trainable = False

In [18]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [19]:
model.fit(train_generator, validation_data=val_generator, epochs=33)

Epoch 1/33


/opt/conda/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


 1/93 ━━━━━━━━━━━━━━━━━━━━ 22:12 14s/step - accuracy: 0.0625 - loss: 1.8687

I0000 00:00:1719115305.598617     664 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


93/93 ━━━━━━━━━━━━━━━━━━━━ 62s 512ms/step - accuracy: 0.3050 - loss: 1.9688 - val_accuracy: 0.3941 - val_loss: 1.3816
Epoch 2/33
93/93 ━━━━━━━━━━━━━━━━━━━━ 26s 263ms/step - accuracy: 0.4155 - loss: 1.3365 - val_accuracy: 0.3900 - val_loss: 1.3781
Epoch 3/33
93/93 ━━━━━━━━━━━━━━━━━━━━ 25s 254ms/step - accuracy: 0.4648 - loss: 1.2450 - val_accuracy: 0.4062 - val_loss: 1.3196
Epoch 4/33
93/93 ━━━━━━━━━━━━━━━━━━━━ 25s 256ms/step - accuracy: 0.4776 - loss: 1.2059 - val_accuracy: 0.3860 - val_loss: 1.3502
Epoch 5/33
93/93 ━━━━━━━━━━━━━━━━━━━━ 26s 265ms/step - accuracy: 0.5047 - loss: 1.1661 - val_accuracy: 0.3887 - val_loss: 1.3554
Epoch 6/33
93/93 ━━━━━━━━━━━━━━━━━━━━ 26s 265ms/step - accuracy: 0.5365 - loss: 1.1135 - val_accuracy: 0.4035 - val_loss: 1.3176
Epoch 7/33
93/93 ━━━━━━━━━━━━━━━━━━━━ 25s 251ms/step - accuracy: 0.5582 - loss: 1.0746 - val_accuracy: 0.4157 - val_loss: 1.3790
Epoch 8/33
93/93 ━━━━━━━━━━━━━━━━━━━━ 26s 259ms/step - accuracy: 0.5700 - loss: 1.0457 - val_accuracy: 0.383

In [1]:
model.save('drdMbNetV2_40epochs_97.h5')

NameError: name 'model' is not defined

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
import numpy as np
from tensorflow.keras.models import load_model

loaded_model = load_model('drdMbNetV2_40epochs_97.h5')
# Assuming you have the loaded_model
def predict_image_class(image_path, loaded_model):
    img = image.load_img(image_path, target_size=(224, 224))  # Adjust target_size as needed
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = img / 255.0  # Normalize the image (assuming the model expects normalized input)

    # Make a prediction
    predictions = loaded_model.predict(img)
    class_index = np.argmax(predictions)
    class_probability = predictions[0][class_index]

    # Replace class_labels with your own list of class labels
    class_labels = ['No_DR','Mild', 'Moderate', 'severe','Proliferative DR']
#     class_labels = ['class0','class1', 'class2', 'class3','class4']  # Update with your actual class labels
    predicted_class = class_labels[class_index]

    return predicted_class, class_probability

image_path=r'/kaggle/input/diabetic-retinopathy-resized/resized_train/resized_train/10013_left.jpeg'

predicted_class, class_probability = predict_image_class(image_path, loaded_model)
print(f'Predicted class: {predicted_class}')
print(f'Class probability: {class_probability:.2f}')


: 

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
import numpy as np
from tensorflow.keras.models import load_model

loaded_model = load_model('drdMbNetV2.h5')
# Assuming you have the loaded_model
def predict_image_class(image_path, loaded_model):
    img = image.load_img(image_path, target_size=(512, 512))  # Adjust target_size as needed
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = img / 255.0  # Normalize the image (assuming the model expects normalized input)

    # Make a prediction
    predictions = loaded_model.predict(img)
    class_index = np.argmax(predictions)
    class_probability = predictions[0][class_index]

    # Replace class_labels with your own list of class labels
    class_labels = ['No_DR','Mild', 'Moderate', 'severe','Proliferative DR']
#     class_labels = ['class0','class1', 'class2', 'class3','class4']  # Update with your actual class labels
    predicted_class = class_labels[class_index]

    return predicted_class, class_probability

image_path=r'test class\class 1\8.png'

predicted_class, class_probability = predict_image_class(image_path, loaded_model)
print(f'Predicted class: {predicted_class}')
print(f'Class probability: {class_probability:.2f}')


1/1 [==============================] - 6s 6s/step
Predicted class: Mild
Class probability: 0.52
